<a href="https://colab.research.google.com/github/Rushikeshaya/Generative-AI/blob/main/Practical_No_06_Retrieval_Augmented_Generation_(RAG)_using_Pre_trained_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Domain-Specific Question Answering Using Retrieval-Augmented Generation (RAG)

**Student Name:** Rushikesh Hande  
**PRN:** 202401110056  
**Platform:** Google Colab  
**Domain:** Academic / Educational Documents

## Project Overview
In this project, I build a Retrieval-Augmented Generation (RAG) application that answers questions from a collection of domain-specific PDF documents. I compare the answers produced by a pretrained Large Language Model (LLM) without retrieval against the answers produced by the same model with retrieved document context.

I use document loading, text cleaning, chunking, sentence embeddings, a FAISS vector database, semantic retrieval, and a pretrained FLAN-T5 model. I also evaluate the outputs using answer similarity and a context-grounding checklist.


## 1. My Objectives

- I load and clean a collection of PDF documents.
- I split the extracted text into overlapping chunks suitable for semantic retrieval.
- I generate dense embeddings using a pretrained Sentence Transformers model.
- I create a FAISS vector index and retrieve the most relevant chunks for each question.
- I generate an answer with FLAN-T5 using retrieved context (RAG).
- I generate a baseline answer using FLAN-T5 without retrieved context (LLM-only).
- I compare both approaches and discuss their strengths, limitations, and possible hallucinations.

**My selected domain is academic / educational documents.** I can upload lecture notes, course handouts, textbooks, or other relevant PDFs below.


In [ ]:
# 2. Install the libraries I used in this project
!pip -q install pypdf sentence-transformers faiss-cpu transformers accelerate rouge-score

## 3. I Upload My Document Collection

I upload one or more PDF files. The files I provide should belong to my chosen domain and contain information that I want the application to answer questions about.


In [ ]:
from google.colab import files

uploaded = files.upload()  # I select my domain-specific PDF documents here.

pdf_paths = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]
if not pdf_paths:
    raise ValueError("I need to upload at least one PDF file.")
print("PDF documents uploaded:", pdf_paths)

Saving 2.Theory_of_Computation(www.gatenotes.in).pdf to 2.Theory_of_Computation(www.gatenotes.in) (1).pdf
PDF documents uploaded: ['2.Theory_of_Computation(www.gatenotes.in) (1).pdf']


## 4. I Load and Clean the PDF Text

I use PyPDF to extract text from each PDF. I normalize whitespace and remove empty text. PDF extraction quality depends on the document; scanned PDFs may require OCR, which is not included in this baseline notebook.


In [ ]:
from pypdf import PdfReader
import re

documents = []

for path in pdf_paths:
    reader = PdfReader(path)
    page_texts = []
    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = re.sub(r"\\s+", " ", text).strip()
        if text:
            page_texts.append({
                "source": path,
                "page": page_num,
                "text": text
            })
    documents.extend(page_texts)

if not documents:
    raise ValueError("No text was extracted. I should check whether my PDFs contain selectable text.")

print(f"I extracted text from {len(documents)} pages.")
print("\\nSample extracted text:")
print(documents[0]["text"][:1000])

I extracted text from 168 pages.
\nSample extracted text:
INTRODUCTION TO THEORY OF COMPUTATIONS 
• As word suggests ‘TOC’ is the study of 'mathematical' machines or systems called 
automata and the computational problems that can and cannot be solved using these 
machines, i.e. what is the extent to which a problem is solvable on a computer. Theory 
of computation can be considered as the study of all kinds of computational model in 
the field of computer science and it also considers how efficiently the problem can be 
solved (but not is depth). 
PROBLEM 
• Now a day’s machines (digital, analog, mechanical) play a very important role in the 
development of human, we need some mechanism (language) to communicate with the 
machines. 
SOLUTION 
• We need a language for communication with machines. 
• But we do not require natural languages to communicate with the machines, as natural 
languages are very complex and machine interaction require very fewer complex 
languages compare to nat

## 5. I Chunk the Documents

I split the page text into chunks of about 700 characters with an overlap of 120 characters. Chunking makes long documents easier to search. The overlap helps preserve information that might otherwise be separated at chunk boundaries. I retain source filename and page number as metadata so that I can show where retrieved information came from.


In [ ]:
CHUNK_SIZE = 700
CHUNK_OVERLAP = 120

chunks = []

for doc in documents:
    text = doc["text"]
    start = 0
    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunk_text = text[start:end].strip()
        if chunk_text:
            chunks.append({
                "text": chunk_text,
                "source": doc["source"],
                "page": doc["page"],
                "chunk_start": start
            })
        if end == len(text):
            break
        start = end - CHUNK_OVERLAP

print(f"I created {len(chunks)} chunks.")
print("Example chunk:")
print(chunks[0])

I created 317 chunks.
Example chunk:
{'text': "INTRODUCTION TO THEORY OF COMPUTATIONS \n• As word suggests ‘TOC’ is the study of 'mathematical' machines or systems called \nautomata and the computational problems that can and cannot be solved using these \nmachines, i.e. what is the extent to which a problem is solvable on a computer. Theory \nof computation can be considered as the study of all kinds of computational model in \nthe field of computer science and it also considers how efficiently the problem can be \nsolved (but not is depth). \nPROBLEM \n• Now a day’s machines (digital, analog, mechanical) play a very important role in the \ndevelopment of human, we need some mechanism (language) to communicate with the \nmachines. \nSOLUTION \n• We", 'source': '2.Theory_of_Computation(www.gatenotes.in) (1).pdf', 'page': 1, 'chunk_start': 0}


## 6. I Generate Embeddings and Build the FAISS Vector Database

I use `all-MiniLM-L6-v2`, a pretrained Sentence Transformers model that maps text into dense vectors. I normalize the vectors and use FAISS inner-product search, which is equivalent to cosine similarity for normalized embeddings. This allows me to retrieve semantically relevant chunks rather than relying only on exact keyword matches.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

chunk_texts = [chunk["text"] for chunk in chunks]
embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

embedding_dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dimension)
index.add(embeddings)

print("I indexed", index.ntotal, "chunks in FAISS.")
print("Embedding dimension:", embedding_dimension)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

I indexed 317 chunks in FAISS.
Embedding dimension: 384


## 7. I Load My Pretrained LLM

I have used `google/flan-t5-base` as the pretrained instruction-following language model. I use the same model for both the baseline and RAG responses so that my comparison focuses on the effect of retrieval. The model is downloaded from Hugging Face when this cell runs.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

LLM_NAME = "google/flan-t5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForSeq2SeqLM.from_pretrained(LLM_NAME).to(device)

print("I loaded", LLM_NAME, "on", device)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


I loaded google/flan-t5-base on cpu


## 8. I Implement Semantic Retrieval

For every question, I encode the question with the same embedding model and search the FAISS index. I retrieve the top-k chunks with the highest similarity scores. I also return their source and page metadata for traceability.


In [ ]:
def retrieve(question, top_k=4):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    k = min(top_k, index.ntotal)
    scores, indices = index.search(question_embedding, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        item = chunks[int(idx)].copy()
        item["similarity_score"] = float(score)
        results.append(item)
    return results

# I test retrieval with an example question.
# I can replace this question with one related to my uploaded documents.
sample_question = "What are the main concepts discussed in these documents?"
retrieved_sample = retrieve(sample_question, top_k=4)

for rank, item in enumerate(retrieved_sample, start=1):
    print(f"\\nRank {rank} | similarity={item['similarity_score']:.3f} | "
          f"source={item['source']} | page={item['page']}")
    print(item["text"][:500])

\nRank 1 | similarity=0.351 | source=2.Theory_of_Computation(www.gatenotes.in) (1).pdf | page=136
Formal Definition of PDA 
A PDA is a 7-tuple, namely (Q, ∑, Γ, δ, q0, Z0, F), where 
(i) Q – is a finite nonempty set of states, 
(ii) ∑ – is a finite nonempty set of input symbols, 
(iii) Γ – is a finite nonempty set of pushdown symbols, 
(iv) q0 – is a special state called the initial state, 
(v) Z0 – is a special pushdown symbol called the initial symbol on the pushdown store. 
(vi) F – is a set of final states, a subset of Q and 
(vii) δ – is a transition function from Q x (∑ U {∈}) x Γ to t
\nRank 2 | similarity=0.316 | source=2.Theory_of_Computation(www.gatenotes.in) (1).pdf | page=137
on (ID) 
 
• An instantaneous description (ID) is (q, x, α), where q ∈ Q, x ∈ ∑* and α ∈ Γ*. 
• An initial ID is (q0, x, Z 0), this means that initially the pda is in the initial state q 0, the 
input string to be processed is x and the PDS has only one symbol, namely Z0.
\nRank 3 | similarity=0.309 | 

## 9. I Generate Answers in Two Modes

### LLM-only baseline
I provide the question directly to FLAN-T5 without document context. The answer may rely on the model's pretrained knowledge and may not reflect the uploaded documents.

### RAG
I retrieve relevant chunks and include them in the prompt. I instruct the model to answer using the supplied context and to state when the context does not contain enough information. This prompt helps reduce unsupported answers, although it cannot guarantee that hallucinations will never occur.


In [ ]:
def generate_text(prompt, max_new_tokens=180):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            do_sample=False
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


def answer_llm_only(question):
    prompt = (
        "Answer the following question using your general knowledge. "
        "If you are unsure, say so.\\n"
        f"Question: {question}\\n"
        "Answer:"
    )
    return generate_text(prompt)


def answer_with_rag(question, top_k=4):
    retrieved = retrieve(question, top_k=top_k)
    context = "\\n\\n".join(
        f"[Source: {item['source']}, page: {item['page']}] {item['text']}"
        for item in retrieved
    )

    prompt = (
        "You are a question-answering assistant. Answer using only the context below. "
        "If the context does not contain enough information, say: "
        "'The provided documents do not contain enough information to answer this.' "
        "Do not invent facts.\\n\\n"
        f"Context:\\n{context}\\n\\n"
        f"Question: {question}\\n"
        "Grounded answer:"
    )
    answer = generate_text(prompt)
    return answer, retrieved

## 10. I Build a Question-Answering Demonstration

I enter a question about my uploaded document collection. The notebook displays the LLM-only answer, the RAG answer, and the retrieved evidence with source metadata.


In [ ]:
question = input("I enter my question about the uploaded documents: ").strip()

if not question:
    raise ValueError("I must enter a question.")

baseline_answer = answer_llm_only(question)
rag_answer, retrieved_chunks = answer_with_rag(question, top_k=4)

print("\\n================ LLM-ONLY ANSWER ================\\n")
print(baseline_answer)

print("\\n================ RAG ANSWER ======================\\n")
print(rag_answer)

print("\\n================ RETRIEVED EVIDENCE ==============\\n")
for i, item in enumerate(retrieved_chunks, start=1):
    print(f"Evidence {i}: {item['source']} | Page {item['page']} | "
          f"Similarity {item['similarity_score']:.3f}")
    print(item["text"])
    print("-" * 80)

I enter my question about the uploaded documents: what is language
\n================ LLM-ONLY ANSWER ================\n
English Language
\n================ RAG ANSWER ======================\n
The concept of defining a language using grammar is, starting from a start symbol using the production rules of the grammar any time, deriving the string. Here every time during derivation a production is used as its LHS is replaced by its RHS, all the intermediate stages(strings) are called sentential forms. The language formed by the grammar consists of all distinct strings that can be generated in this manner. L (G) = w | w  * , S * W • *(reflexive, transitive closure) means from s we can derive w in zero or more steps
\n================ RETRIEVED EVIDENCE ==============\n
Evidence 1: 2.Theory_of_Computation(www.gatenotes.in) (1).pdf | Page 3 | Similarity 0.609
METHODS TO DEFINE LANGUAGE 
• In natural language we define the list of words in a dictionary because they are finite 
and predefined,

## 11. I Evaluate and Compare the Responses

I use ROUGE-L as a lightweight lexical-overlap measure when I have a reference answer. I also record retrieval similarity and conduct a context-grounding review. ROUGE is not a complete measure of correctness or faithfulness; therefore, I interpret it alongside manual inspection.

For a stronger evaluation, I should prepare a set of questions and reference answers based on my documents, then review whether each generated answer is factually supported by the retrieved passages.


In [ ]:
from rouge_score import rouge_scorer
import pandas as pd

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# I may enter a reference answer based on my source documents.
reference_answer = input(
    "Optional: I enter a reference answer for this question (or press Enter to skip): "
).strip()

evaluation_record = {
    "question": question,
    "llm_only_answer": baseline_answer,
    "rag_answer": rag_answer,
    "retrieved_sources": "; ".join(
        f"{item['source']} (page {item['page']})" for item in retrieved_chunks
    ),
    "top_retrieval_similarity": (
        retrieved_chunks[0]["similarity_score"] if retrieved_chunks else None
    ),
    "reference_answer": reference_answer or None
}

if reference_answer:
    baseline_rouge = rouge.score(reference_answer, baseline_answer)["rougeL"].fmeasure
    rag_rouge = rouge.score(reference_answer, rag_answer)["rougeL"].fmeasure
    evaluation_record["llm_only_rougeL"] = baseline_rouge
    evaluation_record["rag_rougeL"] = rag_rouge
    print(f"LLM-only ROUGE-L: {baseline_rouge:.4f}")
    print(f"RAG ROUGE-L:      {rag_rouge:.4f}")
else:
    print("I skipped ROUGE-L because I did not provide a reference answer.")

results_df = pd.DataFrame([evaluation_record])
display(results_df)

Optional: I enter a reference answer for this question (or press Enter to skip): 
I skipped ROUGE-L because I did not provide a reference answer.


,question,llm_only_answer,rag_answer,retrieved_sources,top_retrieval_similarity,reference_answer
0,what is language,English Language,The concept of defining a language using gramm...,2.Theory_of_Computation(www.gatenotes.in) (1)....,0.608638,None


## 12. I Run Evaluation on Multiple Questions

I can prepare a list of test questions. If I also provide reference answers, I can compare ROUGE-L scores across the questions. I use this evaluation as an indicative metric and manually inspect the answers for correctness, relevance, and whether the RAG response is supported by retrieved evidence.


In [ ]:
# I can edit these example questions to fit my uploaded documents.
# Add reference answers when I want to calculate ROUGE-L.
test_cases = [
    {"question": "What is the main topic of the uploaded documents?", "reference": ""},
    {"question": "Explain one important concept described in the documents.", "reference": ""},
    {"question": "What key steps or methods are described?", "reference": ""}
]

evaluation_rows = []

for case in test_cases:
    q = case["question"]
    reference = case["reference"].strip()

    base = answer_llm_only(q)
    rag, retrieved = answer_with_rag(q, top_k=4)

    row = {
        "question": q,
        "llm_only_answer": base,
        "rag_answer": rag,
        "retrieved_sources": "; ".join(
            f"{x['source']} p.{x['page']}" for x in retrieved
        ),
        "top_similarity": retrieved[0]["similarity_score"] if retrieved else None,
        "reference_answer": reference or None
    }

    if reference:
        row["llm_only_rougeL"] = rouge.score(reference, base)["rougeL"].fmeasure
        row["rag_rougeL"] = rouge.score(reference, rag)["rougeL"].fmeasure

    evaluation_rows.append(row)

batch_results = pd.DataFrame(evaluation_rows)
display(batch_results)

,question,llm_only_answer,rag_answer,retrieved_sources,top_similarity,reference_answer
0,What is the main topic of the uploaded documents?,Documents relating to a wide variety of topics.,The provided documents do not contain enough i...,2.Theory_of_Computation(www.gatenotes.in) (1)....,0.242110,None
1,Explain one important concept described in the...,The use of a compass is a way of determining t...,"A PDA is a 7-tuple, namely (Q, , , , , q0, Z0,...",2.Theory_of_Computation(www.gatenotes.in) (1)....,0.365298,None
2,What key steps or methods are described?,The first step in the process is to identify t...,• We will be pushing a’s as they come and when...,2.Theory_of_Computation(www.gatenotes.in) (1)....,0.304217,None


## 13. I Save My Comparative Evaluation Results

I save my comparison table as a CSV file so that I can include the results in my project report. The CSV contains the questions, both generated answers, retrieved source information, and any available evaluation scores.


In [ ]:

display(batch_results)

,question,llm_only_answer,rag_answer,retrieved_sources,top_similarity,reference_answer
0,What is the main topic of the uploaded documents?,Documents relating to a wide variety of topics.,The provided documents do not contain enough i...,2.Theory_of_Computation(www.gatenotes.in) (1)....,0.242110,None
1,Explain one important concept described in the...,The use of a compass is a way of determining t...,"A PDA is a 7-tuple, namely (Q, , , , , q0, Z0,...",2.Theory_of_Computation(www.gatenotes.in) (1)....,0.365298,None
2,What key steps or methods are described?,The first step in the process is to identify t...,• We will be pushing a’s as they come and when...,2.Theory_of_Computation(www.gatenotes.in) (1)....,0.304217,None


## 14. I Analyze the Results

I compare the two approaches using these criteria:

| Criterion | LLM-only | RAG |
|---|---|---|
| Access to my uploaded documents | No | Yes, through retrieval |
| Answers grounded in my collection | Not guaranteed | Improved by retrieved context |
| Domain-specific details | May be missing | Can use document evidence |
| Hallucination risk | May answer from unsupported knowledge | Reduced when context and prompt are effective, but not eliminated |
| Traceability | No document citations | Retrieved source filename and page are displayed |
| Dependence on document quality | Low | High |

### My interpretation
I expect RAG to perform better for questions that require information specific to my uploaded documents, because the model receives relevant text as context. The LLM-only baseline may answer general questions well, but it may not know the particular details in my document collection. The result must be verified using my actual test questions and reference answers; I do not assume that RAG will always outperform the baseline.

### Limitations I identify
- I depend on the quality and text extractability of my PDFs.
- Fixed-size character chunking may split a concept across chunks.
- Similarity retrieval may return passages that are related but not sufficient to answer a question.
- FLAN-T5-base has a limited context window and may generate incomplete answers.
- ROUGE-L measures text overlap and does not fully measure factual correctness or groundedness.
- My RAG prompt reduces, but does not eliminate, hallucinations.
- This baseline does not include OCR for scanned PDFs or a comprehensive automated faithfulness metric.


## 15. Conclusion

In this project, I implemented a domain-specific Retrieval-Augmented Generation system for academic and educational documents. I loaded PDF text, cleaned and chunked it, generated semantic embeddings, indexed the chunks in FAISS, retrieved relevant passages, and used a pretrained FLAN-T5 model to generate grounded answers.

I also implemented an LLM-only baseline using the same pretrained model. By comparing the two response modes with retrieved evidence and optional ROUGE-L scoring, I can investigate how document retrieval affects answer relevance and grounding. My system provides a practical demonstration of how RAG can help a pretrained LLM answer questions about a specific document collection while also showing the limitations that need to be considered.

**Student:** Rushikesh Hande  
**PRN:** 202401110056
